# 🏭 제조 로그 분석 & LLM 활용 — 실습 노트북

**2026 AI+제조 교육 · 9회차** · 알티엠(rtm.ai) 이승희

> 흩어진 **비정형 텍스트 로그**(정비 작업지시서·알람·품질 보고서)를
> LLM 으로 **정규화 → 추출 → 분류 → 요약 → 검색**하는 5대 태스크를 직접 실행합니다.

---

### 🎯 오늘 만드는 것
| 실습 | 태스크 | 결과물 |
|------|--------|--------|
| **실습 1** | 환경세팅 + **① 정규화** | 17가지 표기 → 의미 단위 통합, 집계가 살아난다 |
| **실습 2** | **② NER · ③ 분류 · ④ 요약** | 자유서술 → 구조화 JSON, 자동 태깅, 알람 1,247건 → 3줄 |
| **실습 3** | **⑤ 검색(RAG)** vs **LLM 위키** | 의미검색 + 전역질문 비교 |
| **보너스** | 문서 → MLLM 변환 | 표 이미지·PDF → 구조화 마크다운 |

### 🚦 2-트랙 — 본인 스타일대로 골라 들으세요
- **💻 코드 트랙** — 셀을 직접 실행(Python + `google-genai`). 결과를 표로 받아 파이프라인에 연결.
- **📝 노코드 트랙** — 각 태스크의 <b>복붙 프롬프트</b> 블록을 **익숙한 ChatGPT·Claude·Gemini 채팅**에 붙여넣기.

---
> ⚠️ **데이터 고지** — 이 노트북의 로그 샘플(MWO·알람·MES·NCR·SOP)은 **교육용으로 만든 합성(가상) 데이터**입니다.
> 실제 기업 데이터가 아니며, 정규화 효과를 잘 보여주도록 표기 편차를 의도적으로 심어 두었습니다.


---
# 🔧 실습 1 — 환경세팅 + ① 정규화

**STEP 1** 무료 Gemini API 키 발급 → **STEP 2** Colab 연결 & 호출 준비 → **STEP 3** 로그 데이터 불러오기 → **① 정규화**

## STEP 1 · 무료 Gemini API 키 발급

1. [**aistudio.google.com/apikey**](https://aistudio.google.com/apikey) 접속 (구글 계정 로그인)
2. **`Create API key`** → `Create API key in new project` 클릭
3. 생성된 키(`AIza...`)를 **복사**

> 💡 **무료 티어**로 충분합니다. `gemini-3.5-flash` 기준 분당·일일 요청 한도가 실습에 넉넉합니다.
> 키는 **비밀**입니다. 화면 공유·깃허브에 노출하지 마세요.

## STEP 2 · Colab 연결 & 호출 준비

발급한 키를 Colab 비밀(🔑 Secrets)에 `GEMINI_API_KEY` 로 등록한 뒤, 라이브러리를 설치하고 클라이언트와 공용 호출 헬퍼(`ask`)를 준비합니다.

1. 왼쪽 사이드바 **🔑(보안 비밀)** → **`+ 새 보안 비밀 추가`**
2. 이름 `GEMINI_API_KEY`, 값에 발급한 키(`AIza...`) 붙여넣기 → **노트북 액세스** 토글 ON
3. 아래 세 셀을 차례로 실행 (설치 → 클라이언트 → 헬퍼)

In [ ]:
from google.colab import userdata  # Colab 환경

API_KEY = userdata.get("GEMINI_API_KEY2")

from google import genai
client = genai.Client(api_key=API_KEY)

MODEL = "gemini-3.5-flash"            # 빠르고 멀티모달(이미지·PDF) 지원.
EMBED_MODEL = "gemini-embedding-001"  # 임베딩(실습3)에 사용

# 연결 테스트
r = client.models.generate_content(model=MODEL, contents="한 문장으로 자기소개해줘.")
print("✅ 연결 성공:", r.text.strip())

In [ ]:
# 공용 호출 헬퍼 — 텍스트/JSON 두 가지 모드. 가벼운 재시도 포함(무료 티어 레이트리밋 대비).
import json, time, re
NL, TAB = chr(10), chr(9)   # 실제 줄바꿈/탭 — 프롬프트 조립에 사용(리터럴 백슬래시-n 방지)

def ask(prompt, json_mode=False, model=MODEL, temperature=0.2, retries=3, save_raw=None):
    """Gemini 호출. json_mode=True 면 dict/list 로 파싱해 반환.
    save_raw='파일명' 지정 시 모델 원문(raw 텍스트)을 그 파일로 저장 — JSON 형식이 잘 지켜졌는지 확인용."""
    cfg = {"temperature": temperature}
    if json_mode:
        cfg["response_mime_type"] = "application/json"
    for i in range(retries):
        try:
            resp = client.models.generate_content(model=model, contents=prompt, config=cfg)
            text = resp.text
            if save_raw:
                with open(save_raw, "w", encoding="utf-8") as f:
                    f.write(text)
            return json.loads(text) if json_mode else text.strip()
        except Exception as e:
            if i == retries - 1:
                raise
            time.sleep(2 * (i + 1))  # 백오프

print("ask() 준비 완료 — ask('질문'), ask('...', json_mode=True), ask('...', save_raw='파일명')")

## STEP 3 · 데이터 — 실제 로그 **파일**에서 불러오기

실무 로그는 메모리 안 리스트가 아니라 **파일(.txt/.csv)·DB** 로 존재합니다.
이 실습의 샘플 5종은 **강의 자료의 `data/` 폴더**로 함께 배포됩니다 — 아래 셀은 그 파일을 **읽기만** 합니다.
실무 적용 시엔 **본인 로그 파일**을 같은 이름으로 `data/`(또는 현재 폴더)에 두면 그대로 동작합니다.

| 파일 | 내용 | 건수 |
|------|------|------|
| `mwo.txt` | 정비 작업지시서(자유서술·한영·약어 혼용) | **180** |
| `alarms.txt` | 설비 트립 후 알람 폭주(타임스탬프+코드+메시지) | **1,247** |
| `mes.csv` | MES Lot 진행 이벤트(정형 + NOTE 서술) | 50 |
| `ncr.txt` | 품질 부적합 보고(코드 + 서술) | 15 |
| `sop.md` | 작업표준(RAG/위키 매뉴얼) | 6 |

> **노코드 트랙**은 동일 파일(`mwo.txt`·`alarms.txt`)을 **ChatGPT·Claude·Gemini 채팅** 에 그대로 **첨부**합니다.

In [ ]:
import pandas as pd
from pathlib import Path

# 데이터 위치 — 저장소는 data/, Colab 업로드는 현재 폴더. 둘 다 자동 인식.
DATA = Path("data") if Path("data/mwo.txt").exists() else Path(".")
if not (DATA / "mwo.txt").exists():
    raise FileNotFoundError(
        "데이터 파일을 찾을 수 없습니다. data/ 폴더(또는 현재 폴더)에 "
        "mwo.txt·alarms.txt·mes.csv·ncr.txt·sop.md 를 두세요. "
        "(샘플 재생성: 로컬에서 python3 build_data.py)"
    )

# 파일에서 로그 읽기 — 실무에선 동일 이름으로 본인 파일을 올리면 그대로 동작
mwo = [ln.strip() for ln in open(DATA / "mwo.txt", encoding="utf-8") if ln.strip()]
mwo_df = pd.DataFrame({"id": [f"MWO-{i+1:04d}" for i in range(len(mwo))], "text": mwo})

alarms  = [ln.strip() for ln in open(DATA / "alarms.txt", encoding="utf-8") if ln.strip()]   # 1,247건 (실습2 요약)
mes_df  = pd.read_csv(DATA / "mes.csv")
ncr     = [ln.strip() for ln in open(DATA / "ncr.txt", encoding="utf-8") if ln.strip()]
MWO_SOP = [s.strip() for s in open(DATA / "sop.md", encoding="utf-8").read().split("\n\n") if s.strip()]  # 실습3 RAG/위키

print(f"MWO {len(mwo)}건 · 알람 {len(alarms):,}건 · MES {len(mes_df)}행 · NCR {len(ncr)}건 · SOP {len(MWO_SOP)}개  (from {DATA}/)")
mwo_df.head(8)

### 🗺️ 한눈에 보기 — ① 정규화

표기가 제각각인 정비 기록을 **통제 어휘 1개**로 모아 집계가 살아나는 과정을 입력→출력으로 봅니다. 입력은 `mwo.txt` 실제 줄, 출력은 LLM 결과 예시입니다.

In [ ]:
# 🗺️ 한눈에 보기 다이어그램 — 공용 헬퍼 정의(실습 2·3에서 재사용) + ① 정규화
from IPython.display import HTML, display

L, INK = "#cbd5e1", "#1f2937"
INBG, OUTBG, OUTBD, MIDBG, MIDBD = "#f1f5f9", "#ecfdf5", "#34d399", "#eff6ff", "#60a5fa"

def chip(t, bg="#ffffff", bd="#e2e8f0"):
    return (f'<span style="display:inline-block;background:{bg};border:1px solid {bd};'
            f'border-radius:6px;padding:3px 8px;margin:2px;font-size:13px">{t}</span>')

def arrow(label=""):
    lab = f'<div style="font-size:10px;color:#94a3b8;white-space:nowrap">{label}</div>' if label else ""
    return (f'<div style="display:flex;flex-direction:column;align-items:center;justify-content:center;'
            f'color:#94a3b8;font-size:22px;padding:0 8px">→{lab}</div>')

def io(label, content, bg, bd="#e2e8f0"):
    return (f'<div style="background:{bg};border:1px solid {bd};border-radius:8px;padding:8px 11px;'
            f'min-width:130px;max-width:230px">'
            f'<div style="font-size:10px;color:#64748b;text-transform:uppercase;letter-spacing:.04em;'
            f'margin-bottom:5px">{label}</div>'
            f'<div style="font-size:13px;color:#1f2937;line-height:1.55">{content}</div></div>')

def panel(num, title, sub, body):
    return (f'<div style="border:1px solid {L};border-radius:12px;padding:13px 16px;margin:9px 0;background:#fff">'
            f'<div style="font-weight:700;color:{INK};font-size:15px">{num} {title}'
            f'<span style="font-weight:400;color:#64748b;font-size:12px"> — {sub}</span></div>'
            f'<div style="display:flex;align-items:stretch;flex-wrap:wrap;gap:4px;margin-top:9px">{body}</div></div>')

# ① 정규화 — 표기 제각각 → 표준 1개
p_norm = io("입력 · 표기 제각각",
            chip("C-410 마모, BRG repl") + chip("RDX-2 소착, Bearing 소착 교체") +
            chip("베어링교체, IR 측정") + chip("P-101 異音, 베어링 교체"), INBG) \
       + arrow("통제어휘 매핑") \
       + io("출력 · 표준 용어 1개", chip("베어링 교체", OUTBG, OUTBD), OUTBG, OUTBD)

html = ('<div style="font-family:-apple-system,BlinkMacSystemFont,Segoe UI,sans-serif;max-width:980px">'
        + panel("①", "정규화", "표기 편차 → 통제 어휘 1개로 (집계 가능해짐)", p_norm)
        + '<div style="font-size:11px;color:#94a3b8;margin-top:6px">※ 입력은 mwo.txt 실제 데이터, 출력은 LLM 결과 예시.</div>'
        + '</div>')
display(HTML(html))

## ① 정규화 — 표기 편차로 *집계가 안 되는* 현실

같은 "베어링 교체"가 `BRG repl`, `bearing replaced`, `베어링교체`, `Bearing 소착 교체`로 흩어집니다.
정규화 **전**에는 단순 집계로 1위 작업조차 안 잡힙니다. 직접 확인해 봅시다.

In [ ]:
# [정규화 전] 원문 그대로 빈도 집계 → 표기가 제각각이라 잘게 흩어진다
print(f"정규화 전 — 원문 {len(mwo_df)}건 중 고유 표기 {mwo_df['text'].nunique()}종")
print(mwo_df["text"].value_counts().head(8).to_string())
print()
print("→ 표기 편차로 수백 갈래로 흩어져 '가장 잦은 정비 작업'이 안 잡힌다.")

**💻 코드 트랙** — Gemini API로 180건을 정규화합니다. 같은 데이터를 **두 방식**으로 처리해 비교합니다.

- **방식 A — JSON + Python 집계**: LLM은 *정규화만* 하고(줄별 표준용어 JSON), **건수는 Python(`Counter`)이 정확히** 셉니다 → 숫자가 정확.
- **방식 B — 표 출력(노코드와 동일 프롬프트)**: LLM이 정규화 + 빈도 집계 + 표까지 한 번에 → `gemini.google.com` 채팅 결과와 직접 비교용.

> 180건은 짧아 **단 한 번의 호출**로 충분합니다(배치 불필요). 대형 로그(수천~수만 줄)면 배치·분할이 다시 필요합니다.
> 두 방식의 출력은 각각 **파일로 저장**(`정규화_A_*`, `정규화_B_표.md`)되니, A·B·노코드(chat) 결과를 나란히 비교해 보세요.
> 💡 같은 `gemini-3.5-flash` 라도 **chat(앱)과 API 결과는 다를 수 있습니다** — 앱은 숨은 시스템 프롬프트·thinking·자체 집계가 끼어들고, 특히 *건수 세기를 LLM이 직접* 하기 때문입니다. 핵심은 "숫자 일치"가 아니라 **같은 결론(베어링 교체 1위)** 입니다.

**🅰 방식 A — JSON 정규화 + Python 집계** — LLM은 줄별 표준용어(JSON)만 만들고, 건수는 **Python(`Counter`)이 정확히** 셉니다 → 숫자가 정확.

In [ ]:
# [정규화 · 방식 A] 자유서술 → 표준 조치 용어 리스트(JSON). 180건을 한 번에 호출.
NORM_VOCAB = ["베어링 교체", "씰 교체", "모터 교체", "임펠러 교체", "커플링 점검",
              "얼라인먼트 조정", "윤활/그리스 보충", "벨트 장력 조정", "필터 교체",
              "유압유 보충", "진동 측정", "절연저항 측정", "케이블 점검", "알람 리셋"]

def normalize_all(texts):
    """180건을 한 콜로 정규화 → 줄별 표준용어 JSON. (대형 로그면 배치/분할 필요)"""
    body = NL.join(f"{i}{TAB}{t}" for i, t in enumerate(texts))
    prompt = f"""당신은 제조 정비 기록 분석 전문가입니다.
아래 '번호<탭>원문' 각 줄을 표준 조치 용어로 정규화하세요.
- 반드시 다음 표준 용어 중에서만 고르세요: {NORM_VOCAB}
- 한 건에 조치가 여러 개면 모두 포함. 목록에 없으면 "기타: <간단설명>".
- 원문에 없는 내용을 추정하지 마세요.
출력: JSON 배열  [{{"i": 0, "actions": ["용어1"]}}, ...]   번호 i 를 그대로 유지. JSON만.

[MWO]
{body}"""
    arr = ask(prompt, json_mode=True, save_raw="정규화_A_raw.json")   # 원문(raw) 저장 → 형식 확인용
    by_i = {d.get("i", j): d.get("actions", []) for j, d in enumerate(arr)}
    return [by_i.get(i, []) for i in range(len(texts))]

texts = list(mwo_df["text"])
mwo_df["actions"] = normalize_all(texts)        # 180건 → 1콜
print(f"방식 A 완료 — {len(texts)}건 정규화(1콜) · 원문 저장: 정규화_A_raw.json (JSON 형식 확인용)")
mwo_df[["text", "actions"]].head(15)

In [ ]:
# [정규화 후 · 방식 A 집계] 의미 단위로 다시 집계 → 진짜 1위 작업이 드러난다
from collections import Counter
cnt = Counter(a for acts in mwo_df["actions"] for a in acts)
after = pd.DataFrame(cnt.most_common(), columns=["표준 조치", "건수"])
after.to_csv("정규화_A_집계.csv", index=False, encoding="utf-8-sig")   # 방식 B 표·노코드와 비교용
print("정규화 후 — 조치별 빈도(Python Counter 정확 집계):")
print(after.to_string(index=False))
print()
print("확인용 저장: 정규화_A_집계.csv")
print("✅ '베어링 교체'가 압도적 1위로 드러난다. 정규화 전에는 보이지 않던 패턴.")

**🅱 방식 B — 표 출력(노코드와 동일 프롬프트)** — 위 방식 A는 LLM이 정규화만 하고 Python이 집계했습니다.
이번엔 **노코드 트랙과 똑같은 프롬프트**(아래 셀)를 API로 보내, LLM이 정규화·집계·표 작성까지 한 번에 하게 합니다.
방식 A(`정규화_A_집계.csv`)·방식 B(`정규화_B_표.md`)·`gemini.google.com` 채팅 결과를 나란히 비교해 보세요.

In [ ]:
# [정규화 · 방식 B] 노코드 트랙과 '동일한 프롬프트'를 API로 — 180건 한 번에 주고 LLM이 표까지 작성
all_text = NL.join(mwo_df["text"])
norm_table = ask(f"""당신은 제조 정비 기록 분석 전문가입니다.
아래 정비 작업지시서(MWO) {len(mwo_df)}건이 있습니다.
각 줄을 아래 [표준 조치 용어] 중에서만 골라 정규화하고,
조치별 빈도를 집계해 빈도 내림차순 표(표준조치 | 건수 | 해당 원문 예시)로 출력하세요.

[표준 조치 용어]
{", ".join(NORM_VOCAB)}

- 한 건에 조치가 여러 개면 모두 포함. 목록에 없으면 "기타: <간단설명>" 으로 표기.
- 같은 의미면 표기가 달라도 하나로 묶으세요. (예: BRG repl = bearing replaced = 베어링 교체)
- 원문에 없는 내용은 추정하지 마세요.

[MWO {len(mwo_df)}건]
{all_text}""", save_raw="정규화_B_표.md")   # 모델이 만든 표 원문 저장

print("방식 B 완료 — LLM이 집계한 표 · 저장: 정규화_B_표.md")
print("⚠️ 건수는 LLM이 직접 센 값이라 방식 A(Python)와 다를 수 있습니다 — 두 파일을 비교해 보세요.")
from IPython.display import Markdown, display
display(Markdown(norm_table))

**📝 노코드 트랙** — 코드 없이 같은 결과. **ChatGPT·Claude·Gemini 채팅**에서 **📎(파일 첨부)로 `mwo.txt` 를 올린 뒤** 아래 지시문을 입력하세요.

```text
당신은 제조 정비 기록 분석 전문가입니다.
첨부한 mwo.txt 의 각 줄은 정비 작업지시서(MWO) 한 건(총 180건)입니다.
각 줄을 아래 [표준 조치 용어] 중에서만 골라 정규화하고, 조치별 빈도를 집계해
빈도 내림차순 표(표준조치 | 건수 | 해당 원문 예시)로 출력하세요.

[표준 조치 용어]
베어링 교체, 씰 교체, 모터 교체, 임펠러 교체, 커플링 점검, 얼라인먼트 조정,
윤활/그리스 보충, 벨트 장력 조정, 필터 교체, 유압유 보충, 진동 측정,
절연저항 측정, 케이블 점검, 알람 리셋

- 한 건에 조치가 여러 개면 모두 포함. 목록에 없으면 "기타: <간단설명>" 으로 표기.
- 같은 의미면 표기가 달라도 하나로 묶으세요. (예: BRG repl = bearing replaced = 베어링 교체)
- 원문에 없는 내용은 추정하지 마세요.
```

> 💡 위 [표준 조치 용어] 14종은 코드 트랙의 `NORM_VOCAB`와 같습니다.
> 두 트랙 모두 **"베어링 교체"가 1위**라는 같은 결론에 도달합니다.

---
# 🔧 실습 2 — ② NER · ③ 분류 · ④ 요약

정규화된 로그에서 한 걸음 더: **구조화 추출(NER) → 자동 태깅(분류) → 알람 폭주 압축(요약)**.

### 좋은 프롬프트 4요소 (모든 셀에 공통 적용)
| 요소 | 예 |
|------|----|
| **역할** | "당신은 정비 기록 분석 전문가" |
| **입력 정의** | "MWO 한 건" |
| **출력 스키마** | JSON 필드 명시 (part, symptom, action, location) |
| **제약** | "원문에 없는 건 추정하지 말 것" |

### 🗺️ 한눈에 보기 — ② NER · ③ 분류 · ④ 요약

같은 정비/알람 로그가 세 태스크에서 어떻게 바뀌는지 입력→출력으로 봅니다. 입력은 실제 데이터(`mwo.txt`·`alarms.txt`) 줄, 출력은 LLM 결과 예시입니다.

In [ ]:
# 🗺️ 한눈에 보기 — ② NER · ③ 분류 · ④ 요약  (실습 1의 다이어그램 셀에서 정의한 헬퍼 재사용)
# ② NER — 자유 문장 → 구조화 필드 (mwo.txt 1행)
ner_out = ("<b>part</b> 기계씰<br><b>symptom</b> 진동 상승<br>"
           "<b>action</b> 얼라인먼트 조정 · 씰 교체<br><b>location</b> FAN-12")
p_ner = io("입력 · 자유 문장", "FAN-12 진동 상승,<br>얼라인 재조정, 기계씰 교체", INBG) \
      + arrow("필드 추출") + io("출력 · 구조화 JSON", ner_out, OUTBG, OUTBD)

# ③ 분류 — 로그 → 라벨
cls_out = "<b>장애유형</b> 기계<br><b>심각도</b> 상 <span style='color:#64748b'>(소착=가동중단)</span><br><b>책임공정</b> 회전기계"
p_cls = io("입력 · 한 줄 로그", "RDX-2 소착,<br>Bearing 소착 교체", INBG) \
      + arrow("라벨 태깅") + io("출력 · 태그 3종", cls_out, OUTBG, OUTBD)

# ④ 요약 — 알람 폭주 → 3줄 인과
sum_out = ("① 근본원인: VFD 저전압→모터 과부하(추정)<br>"
           "② 전개: 전압강하 → 펌프 진동·베어링 과열 → 트립·E-Stop<br>"
           "③ 권고: 전원 안정화·과열 인터록 점검")
p_sum = io("입력 · 알람 폭주 1,247건",
           chip("ALM-1180 VFD 저전압") + chip("ALM-2044 P-204 진동 high-high") +
           chip("ALM-2045 베어링 과열") + chip("ALM-3021 모터 과부하 트립") + chip("… 1,247건"), INBG) \
      + arrow("인과 3줄 압축") \
      + io("출력 · 3줄 인과 요약", sum_out, OUTBG, OUTBD)

html = ('<div style="font-family:-apple-system,BlinkMacSystemFont,Segoe UI,sans-serif;max-width:980px">'
        + panel("②", "NER", "자유 문장 → part·symptom·action·location 필드", p_ner)
        + panel("③", "분류", "로그 한 줄 → 장애유형·심각도·책임공정 라벨", p_cls)
        + panel("④", "요약", "알람 폭주 1,247건 → 인과 중심 3줄", p_sum)
        + '<div style="font-size:11px;color:#94a3b8;margin-top:6px">※ 입력은 mwo.txt·alarms.txt 실제 데이터, 출력은 LLM 결과 예시.</div>'
        + '</div>')
display(HTML(html))

## ② NER — 자유 문장 → 구조화 JSON

정규식이 못 잡는 **부품·증상·조치·위치**를 한 프롬프트로 추출합니다. 프롬프트 한 장 = 수작업 라벨링 수천 건 효과.

In [ ]:
def ner_batch(texts):
    """여러 MWO에서 한 번에 추출 → 줄별 구조화 JSON 배열. (정규화처럼 1콜)"""
    body = NL.join(f"{i}{TAB}{t}" for i, t in enumerate(texts))
    prompt = f"""당신은 제조 정비 기록 분석 전문가입니다.
아래 '번호<탭>원문' 각 줄에서 정보를 추출해 JSON 배열로 출력하세요.
필드: i(번호 그대로 유지), part(부품/설비), symptom(증상/현상), action(조치), location(위치/설비번호)
- 해당 정보가 없으면 빈 문자열 "".
- 원문에 없는 내용을 추정하지 마세요. JSON만 출력.

[입력]
{body}"""
    arr = ask(prompt, json_mode=True, save_raw="NER_raw.json")   # 원문(raw) 저장 → 형식 확인용
    by_i = {d.get("i", j): d for j, d in enumerate(arr)}
    return [by_i.get(i, {}) for i in range(len(texts))]

# 데모는 앞 12건을 '한 번에' 호출(전체 180건도 정규화처럼 동일 적용 가능). 노코드 트랙도 같은 12건.
SAMPLE = mwo_df.head(12)
ner_rows = ner_batch(list(SAMPLE["text"]))
ner_df = pd.DataFrame(ner_rows).reindex(columns=["part", "symptom", "action", "location"])
ner_df.insert(0, "원문", list(SAMPLE["text"]))
ner_df.to_csv("NER_결과.csv", index=False, encoding="utf-8-sig")   # 결과 저장
print(f"NER 완료 — {len(ner_df)}건 1콜 · 원문 저장: NER_raw.json · 전체 결과: NER_결과.csv")
ner_df

**📝 노코드 트랙 (NER)** — 코드 트랙과 **같은 12건**(mwo.txt 앞 12줄)입니다. 채팅에 `mwo.txt` 를 **첨부**하면 전체를 처리합니다(빠르게 확인만 하려면 아래 12줄만 붙여넣어도 됩니다). ChatGPT·Claude·Gemini 채팅에서:

```text
당신은 제조 정비 기록 분석 전문가입니다.
첨부한 mwo.txt 의 각 줄(파일 미첨부 시 아래 12줄)에서 정보를 추출해 표로 출력하세요.
필드: part(부품/설비), symptom(증상/현상), action(조치), location(위치/설비번호)
- 해당 정보가 없으면 빈칸.
- 원문에 없는 내용을 추정하지 마세요.
표 형식: 원문 | part | symptom | action | location

[MWO 12건 (파일 미첨부 시)]
- FAN-12 진동 상승, 얼라인 재조정, 기계씰 교체
- 커플링점검
- C-410 마모, BRG repl
- coupling check, hyd oil refill
- P-305 트립, filter 교체, greasing
- RDX-2 소착, Bearing 소착 교체
- coupling check, 정렬 조정
- CV-03 異音, 유압유 보충
- 벨트 장력 조정
- RDX-2 과열, 씰교체
- 임펠러교체
- CV-03 과열, 축정렬
```

> 💡 코드 트랙과 **같은 4필드(part·symptom·action·location)·같은 지시문**입니다. 코드는 JSON으로, 노코드는 표로 받습니다.

## ③ 분류 — 장애유형·심각도·책임공정 자동 태깅

같은 12건을 **두 번**(제로샷·퓨샷) 분류해 비교합니다.

- **Zero-shot** — 라벨 정의만 주고 예시 없이 분류
- **Few-shot** — 애매한 경계 사례용 예시 2개를 추가

> 💡 같은 모델·같은 입력인데 **예시 2줄**이 경계 사례의 판단을 어떻게 바꾸는지 보세요. 미리보기 표에서 `A → B ⚠` 표시가 제로샷→퓨샷에서 바뀐 셀입니다. 실무에서는 **Zero-shot 1차로 빠르게 → 흔들리는 항목만 Few-shot 예시로 고정**하는 흐름이 비용·정확도 모두 유리합니다.

In [ ]:
def classify_all(texts, fewshot=True):
    """여러 MWO를 한 번에 분류 → 줄별 JSON 배열(1콜).
    fewshot=False 면 예시 없이 제로샷으로 분류한다."""
    body = NL.join(f"{i}{TAB}{t}" for i, t in enumerate(texts))
    # 경계 사례를 고정하는 예시 2개 — fewshot=True 일 때만 프롬프트에 포함
    examples = ""
    if fewshot:
        examples = (
            NL + "# 참고 예시(Few-shot)" + NL +
            '"E-Stop 작동 라인 정지" -> {"i": 0, "fault_type":"제어","severity":"상","process":"컨베이어"}' + NL +
            '"윤활유 보충" -> {"i": 1, "fault_type":"기계","severity":"하","process":"회전기계"}' + NL
        )
    prompt = f"""당신은 제조 설비 신뢰성 분석가입니다.
아래 '번호<탭>원문' 각 줄(MWO)을 분류해 JSON 배열로 출력하세요.
필드: i(번호 그대로 유지), fault_type(기계|전기|유압|제어 중 하나),
      severity(상|중|하; 가동중단·안전 위험이면 '상'), process(회전기계|컨베이어|사출|유틸리티 중 추정 1개)
근거 없는 추정 최소화. JSON만 출력.
{examples}
[입력]
{body}"""
    tag = "few" if fewshot else "zero"
    arr = ask(prompt, json_mode=True, save_raw=f"분류_{tag}_raw.json")
    by_i = {d.get("i", j): d for j, d in enumerate(arr)}
    return [by_i.get(i, {}) for i in range(len(texts))]

texts = list(SAMPLE["text"])                       # 위 NER 과 같은 샘플 12건
cols  = ["fault_type", "severity", "process"]
zero_df = pd.DataFrame(classify_all(texts, fewshot=False)).reindex(columns=cols)  # 제로샷
few_df  = pd.DataFrame(classify_all(texts, fewshot=True )).reindex(columns=cols)  # 퓨샷

# 제로샷 vs 퓨샷 비교표 — 바뀐 셀만 'A → B ⚠'
rows = []
for idx, t in enumerate(texts):
    rec = {"원문": t}
    for c in cols:
        z, f = zero_df.at[idx, c], few_df.at[idx, c]
        rec[c] = z if z == f else f"{z} → {f} ⚠"
    rows.append(rec)
cmp_df = pd.DataFrame(rows)
cmp_df.to_csv("분류_제로샷vs퓨샷.csv", index=False, encoding="utf-8-sig")   # 전체 비교 저장

n_diff = int((zero_df.values != few_df.values).sum())
print(f"전체 {len(texts)}건 분류 완료(제로샷·퓨샷 각 1콜)")
print(f"제로샷→퓨샷에서 바뀐 셀: {n_diff}개 / 총 {zero_df.size}개 "
      f"(예시 2줄이 경계 사례를 고정 — 0이면 이 샘플은 이미 명확하다는 뜻)")
display(cmp_df)

# 이후 단계는 퓨샷 결과를 최종으로 사용 (전체 180건)
cls_df = few_df.copy()
cls_df.insert(0, "원문", texts)
print()
print("심각도 분포(퓨샷·전체):"); print(cls_df["severity"].value_counts().to_string())
print()
print("장애유형 분포(퓨샷·전체):"); print(cls_df["fault_type"].value_counts().to_string())

**📝 노코드 트랙 (분류)** — ChatGPT·Claude·Gemini 채팅에 `mwo.txt` 를 **첨부**하면 전체, 빠른 확인은 아래 12줄.

```text
당신은 제조 설비 신뢰성 분석가입니다.
첨부한 mwo.txt 의 각 줄(파일 미첨부 시 아래 12줄)을 분류해 표로 출력하세요.
열: 원문 | 장애유형(기계/전기/유압/제어) | 심각도(상/중/하) | 책임공정(회전기계/컨베이어/사출/유틸리티)
- 가동중단·안전위험이면 심각도 '상'.
- 근거 없는 추정은 최소화.

# 참고 예시(Few-shot)
"E-Stop 작동 라인 정지" -> 제어 / 상 / 컨베이어
"윤활유 보충" -> 기계 / 하 / 회전기계

마지막에 심각도·장애유형 분포 요약도 덧붙이세요.

[MWO 12건 (파일 미첨부 시)]
- FAN-12 진동 상승, 얼라인 재조정, 기계씰 교체
- 커플링점검
- C-410 마모, BRG repl
- coupling check, hyd oil refill
- P-305 트립, filter 교체, greasing
- RDX-2 소착, Bearing 소착 교체
- coupling check, 정렬 조정
- CV-03 異音, 유압유 보충
- 벨트 장력 조정
- RDX-2 과열, 씰교체
- 임펠러교체
- CV-03 과열, 축정렬
```

> 💡 코드 트랙(`classify_all`)과 **같은 3필드·같은 Few-shot 예시 2개**. 코드는 JSON, 노코드는 표로 받습니다.
> 위 `# 참고 예시(Few-shot)` 두 줄을 **빼고** 한 번 더 돌려 보세요 — 그게 코드 트랙의 **제로샷**이고, 경계 사례에서 판단이 어떻게 흔들리는지 보입니다.

### ③ 확장 · 같은 기법, 다른 로그 — NCR 품질 부적합 분류

분류 프롬프트는 **데이터 유형이 바뀌어도 그대로** 통합니다. 같은 방식을 **품질 부적합 보고(NCR, `ncr.txt` 15건)** 에 적용해 봅니다.
MWO(정비)와 필드만 바꾸면(원인영역·즉시조치 여부) 끝 — *프롬프트 재사용성*이 LLM 분류의 핵심 장점입니다.

In [ ]:
# NCR 한 건 형식: "NCR-0455 | 외관불량 | 서술..."  → 구조화 분류(15건 1콜)
def classify_ncr_all(lines):
    body = NL.join(f"{i}{TAB}{x}" for i, x in enumerate(lines))
    prompt = f"""당신은 제조 품질 엔지니어입니다.
아래 '번호<탭>원문' 각 줄(품질 부적합 보고 NCR)을 분류해 JSON 배열로 출력하세요.
필드: i(번호 그대로 유지), defect(불량 유형, 원문 기반), cause_area(금형|공정조건|원료|설비 중 추정 1개),
      severity(상|중|하), action_required(즉시 조치 필요 여부 true/false)
원문에 없는 내용은 추정하지 마세요. JSON만 출력.

[입력]
{body}"""
    arr = ask(prompt, json_mode=True, save_raw="NCR분류_raw.json")   # 원문(raw) 저장 → 형식 확인용
    by_i = {d.get("i", j): d for j, d in enumerate(arr)}
    return [by_i.get(i, {}) for i in range(len(lines))]

ncr_rows = classify_ncr_all(ncr)   # 15건 1콜
ncr_cls = pd.DataFrame(ncr_rows).reindex(columns=["defect", "cause_area", "severity", "action_required"])
ncr_cls.insert(0, "NCR", ncr)
display(ncr_cls)
print("원인영역 분포:"); print(ncr_cls["cause_area"].value_counts().to_string())
print("즉시조치 필요:", int(ncr_cls["action_required"].fillna(False).astype(bool).sum()), "/", len(ncr_cls))

**📝 노코드 트랙 (NCR 분류)** — ChatGPT·Claude·Gemini 채팅에 **`ncr.txt` 를 첨부**하고:

```text
당신은 제조 품질 엔지니어입니다.
첨부한 ncr.txt 의 각 줄은 품질 부적합 보고(NCR) 한 건입니다. 각 줄을 분류해 표로 출력하세요.
열: NCR | 불량유형(원문 기반) | 원인영역(금형/공정조건/원료/설비) | 심각도(상/중/하) | 즉시조치필요(O/X)
- 원문에 없는 내용은 추정하지 마세요.
마지막에 원인영역 분포와 즉시조치 필요 건수 요약도 덧붙이세요.
```

> 💡 정비 로그(MWO)에서 쓴 분류 프롬프트를 **필드만 바꿔**(원인영역·즉시조치) 품질 로그(NCR)에 재사용했습니다 — 코드 트랙 `classify_ncr_all` 과 같은 필드·같은 지시문입니다.

## ④ 요약 — 알람 폭주 1,247건 → 3줄 인과 요약

설비 트립 직후 30분간 알람이 **1,247건** 폭주합니다(`alarms.txt`). 운영자는 핵심 인과를 못 봅니다.
**코드 전처리(분포 집계) + LLM(인과 3줄)** 를 결합해 RCA 첫 가설을 즉시 세우게 합니다.

In [ ]:
# alarms.txt 읽기 — 실제 1,247건
alarms = [ln.strip() for ln in open(DATA / "alarms.txt", encoding="utf-8") if ln.strip()]
print(f"총 {len(alarms):,}건 알람")

# (전처리) 코드별 분포 — 무엇이 폭주했는지 먼저 코드로 집계
codes = [a.split("|")[1].strip() for a in alarms]
print()
print("알람 코드 분포 TOP:")
print(pd.Series(codes).value_counts().to_string())
print()
print("처음 7건(인과 시퀀스 초입):")
print(NL.join(alarms[:7]))

In [ ]:
# LLM: 1,247건 전체를 읽고 인과 중심 3줄로 압축
summary = ask(f"""당신은 제조 운영 RCA 전문가입니다.
아래는 설비 트립 직후 30분간 발생한 알람 로그 {len(alarms)}건입니다.
**인과 중심 3줄 요약**으로 압축하세요.
형식:
1) 근본 원인(추정):
2) 전개(인과 순서):
3) 권고 조치:
원문에 없는 사실은 '(추정)' 으로 명시.

[알람 로그 — {len(alarms)}건]
{chr(10).join(alarms)}""")
print(summary)

**📝 노코드 트랙 (요약)** — ChatGPT·Claude·Gemini 채팅에 **`alarms.txt` 를 첨부**하고:

```text
첨부한 alarms.txt 는 설비 트립 직후 30분간 발생한 알람 1,247건입니다.
당신은 제조 운영 RCA 전문가입니다. 인과 중심 3줄(① 근본원인 추정 ② 전개 인과순서 ③ 권고조치)로 압축해 주세요.
원문에 없는 사실은 '(추정)'으로 표시.
```

### 🧩 미니 도전
- 요약 프롬프트에 "운영자 즉시 점검 항목 체크리스트"를 추가로 요청해 보세요.


---
### 🧩 종합 도전 — MES 운영 리포트 (단계 구분 없이 한 번에)

지금까지 **②추출 · ③분류 · ④요약**을 따로 배웠습니다. 이번엔 **단계를 나누지 말고**, 정형 컬럼 + 자유서술이 섞인 MES(`mes.csv` 50행)를 통째로 던져 **운영 리포트 한 장**을 직접 만들어 보세요.

> 💡 데이터가 작을 땐(수십 행) 굳이 추출→분류→집계로 쪼개지 않아도 **잘 짠 프롬프트 하나**가 그 일을 다 합니다. (대형 로그면 정규화 **방식 A**처럼 단계 분할이 다시 필요 — "쪼갤까 / 한 방에" 판단이 핵심)

**목표 리포트**: ① 정지(HOLD) 사유 표준 카테고리별 집계 · ② 공정별 정지 횟수 · ③ 가장 자주 멈춘 공정과 추정 원인 · ④ 권고 조치 2가지
아래 코드는 **출발점**입니다 — 프롬프트를 본인 현장 질문에 맞게 바꿔 보세요.

In [ ]:
# 종합: MES 50행을 통째로 LLM에 → 추출·분류·집계·요약을 한 번에 (단계 분할 없음)
from IPython.display import Markdown, display
mes_csv = mes_df.to_csv(index=False)
report = ask(f"""당신은 제조 MES 운영 분석가입니다.
아래 MES 이벤트 로그(CSV; ts,lot,step,event,note)로 **운영 리포트(마크다운)**를 작성하세요.
- event=HOLD 는 정지, note 는 자유서술 정지/재개 사유입니다.
- note 의 사유는 표준 카테고리(원료수급/설비·금형/품질/기타)로 묶어 집계하세요.
포함:
## 1. 정지 사유 카테고리별 건수
## 2. 공정(step)별 정지 횟수
## 3. 가장 자주 멈춘 공정과 추정 원인
## 4. 권고 조치 2가지
로그에 근거해서만 작성하세요. 원문에 없는 사실은 '(추정)' 으로 표시하세요.

[MES 로그]
{mes_csv}""")
display(Markdown(report))

**📝 노코드 트랙 (종합)** — ChatGPT·Claude·Gemini 채팅에 **`mes.csv` 를 첨부**하고:

```text
당신은 제조 MES 운영 분석가입니다.
첨부한 mes.csv(컬럼: ts, lot, step, event, note)를 분석해 운영 리포트를 작성하세요.
event=HOLD 는 정지, note 는 자유서술 정지/재개 사유입니다. note 사유는 표준 카테고리(원료수급/설비·금형/품질/기타)로 묶어 집계하세요.
1) 정지 사유 카테고리별 건수
2) 공정(step)별 정지 횟수
3) 가장 자주 멈춘 공정과 추정 원인
4) 권고 조치 2가지
로그에 근거해서만 작성하세요. 원문에 없는 사실은 '(추정)' 으로 표시하세요.
```

> 💡 추출·분류·요약을 **한 프롬프트**가 통합 수행 — 코드 트랙과 같은 데이터·같은 목표입니다. 작은 표는 이렇게 한 번에, 큰 로그는 단계 분할로.

---
# 🔧 실습 3 — ⑤ 검색: RAG vs LLM 위키

> **RAG 만이 답이 아니다.** 데이터 규모·질문 유형에 따라 고른다.
> - **벡터 RAG** — 특정 사실 검색에 강함("P-204 베어링 異音 조치는?")
> - **LLM 위키 / 지식 증류** — 전역·집계 질문에 강함("이번 달 최빈 고장은?") ⭐트렌드

두 방식을 같은 데이터로 직접 비교합니다.

### 🗺️ 한눈에 보기 — ⑤ 검색 (RAG)

질문 → 의미검색으로 찾은 청크를 **프롬프트에 끼워 넣어** LLM 입력으로 → 근거 있는 답변. 입력은 `sop.md` 실제 내용, 출력은 LLM 결과 예시입니다.

In [ ]:
# 🗺️ 한눈에 보기 — ⑤ 검색(RAG)  (실습 1의 다이어그램 셀에서 정의한 헬퍼 재사용)
# ⑤ RAG — 검색된 청크를 '프롬프트 안에 끼워 넣어' LLM 입력으로 만든다 (sop.md)
def prompt_box():
    inserted = ('<div style="background:#fef3c7;border:1px dashed #f59e0b;border-radius:6px;'
                'padding:5px 7px;margin:5px 0">'
                '<div style="font-size:9.5px;color:#b45309;margin-bottom:2px">🔍 의미검색으로 채워진 부분 (top-k 청크)</div>'
                '<span style="color:#92400e">SOP-펌프베어링 — 진동 측정 → 윤활 확인 →<br>SKF 6308 규격 교체 → 얼라인먼트 정렬 → 시운전</span>'
                '</div>')
    body = ('<div style="color:#475569">다음 <b>참고자료</b>만 근거로 답하세요.</div>'
            '<div style="color:#475569;margin-top:3px">[참고자료]</div>' + inserted +
            '<div style="color:#475569">[질문] P계열 펌프 베어링 異音,<br>조치 순서는?</div>')
    return ('<div style="background:{0};border:1px solid {1};border-radius:8px;padding:8px 11px;'
            'min-width:250px;max-width:320px">'
            '<div style="font-size:10px;color:#64748b;text-transform:uppercase;letter-spacing:.04em;'
            'margin-bottom:5px">LLM 입력 · 숨은 프롬프트(검색 청크 삽입)</div>'
            '<div style="font-size:12px;line-height:1.55;font-family:ui-monospace,Menlo,monospace">{2}</div>'
            '</div>').format(MIDBG, MIDBD, body)

p_rag = io("질문 (사용자)", "P계열 펌프 베어링<br>異音, 조치 순서는?", INBG) \
      + arrow("의미검색 top-k") \
      + prompt_box() \
      + arrow("이 프롬프트로 호출") \
      + io("LLM 답변 · 근거 기반", "진동 측정 → 윤활 상태 확인 →<br>SKF 6308 베어링 교체 →<br>얼라인먼트 재정렬 → 시운전", OUTBG, OUTBD)

html = ('<div style="font-family:-apple-system,BlinkMacSystemFont,Segoe UI,sans-serif;max-width:980px">'
        + panel("⑤", "검색(RAG)", "검색한 청크를 프롬프트에 끼워 넣어 LLM 입력으로 → 근거 있는 답변", p_rag)
        + '<div style="font-size:11px;color:#94a3b8;margin-top:6px">※ 입력은 sop.md 실제 데이터, 출력은 LLM 결과 예시. 노란 블록이 검색으로 채워져 프롬프트에 삽입되는 부분.</div>'
        + '</div>')
display(HTML(html))

## 5-1 · 벡터 RAG — 임베딩 + 의미검색

문서를 임베딩 → 질문과 코사인 유사도 top-k → 그 근거로 답변 생성. (벡터DB 없이 numpy 로 원리만)

> 🔀 **임베딩은 아래 두 버전 중 *하나만* 실행하세요.** 이후 검색 셀은 어느 쪽을 실행했든 동일 코드로 동작하되, **코퍼스 범위가 달라 결과도 그에 따라 달라집니다.**
> - **버전 1 (라이브 임베딩)** — MWO 일부(`[20:110]`, 90건) + SOP + 질문을 Gemini 로 **직접 호출**해 *한 번에* 임베딩. 임베딩 생성 과정을 보여주는 시연용.
> - **버전 2 (사전 임베딩 로드)** — **전체** MWO(180건) + SOP 를 미리 임베딩해 둔 `corpus_vec.npy`·`q_vec.npy`(질문 q1·q2)를 로드. **API 호출이 없어** 빠르고 무료 한도·네트워크 걱정이 없으며 코퍼스도 전체라 검색 품질이 더 좋습니다.

### 🅐 버전 1 — 라이브 임베딩 (MWO `[20:110]` + SOP)

Gemini 임베딩 API 를 **직접 호출**해 코퍼스 + 질문을 **한 번의 호출**로 모두 임베딩합니다. 임베딩이 실제로 어떻게 만들어지는지 보여주는 시연용 셀입니다(분량을 줄여 90건만 사용).

In [ ]:
import numpy as np

# 코퍼스 = MWO 기록 일부(인덱스 20~110, 90건) + 작업표준 SOP 6개
CORPUS = list(mwo_df["text"])[20:110] + MWO_SOP

# 이후 검색에 쓸 질문 2개
q1 = "P-204 펌프에서 베어링 異音이 날 때 표준 조치 절차는?"            # 특정 사실형 → RAG 강점
q2 = "지금까지 기록 중 가장 빈번한 정비 작업은 무엇이고 몇 건인가?"      # 전역/집계형 → 5-2 에서 비교
QUESTIONS = [q1, q2]

def embed(texts, batch=100):
    """Gemini 임베딩 — 한 번에 최대 100건씩 배치 호출."""
    vecs = []
    for s in range(0, len(texts), batch):
        res = client.models.embed_content(model=EMBED_MODEL, contents=texts[s:s+batch])
        vecs += [e.values for e in res.embeddings]
    return np.array(vecs)

# 코퍼스 + 질문을 '한 번에' 임베딩한 뒤 분리 (같은 모델로 임베딩되어야 비교 가능)
all_vec = embed(CORPUS + QUESTIONS)
corpus_vec, q_vec = all_vec[:len(CORPUS)], all_vec[len(CORPUS):]

def embed_query(question):
    """질문 임베딩 — 미리 만든 q_vec 에서 조회(버전 2 와 동일 인터페이스)."""
    return q_vec[QUESTIONS.index(question)]

print(f"[버전 1] 코퍼스 {len(CORPUS)}개 + 질문 {len(QUESTIONS)}개 임베딩 완료 (dim={corpus_vec.shape[1]})")

### 🅑 버전 2 — 사전 임베딩 로드 (`corpus_vec.npy` · `q_vec.npy`)

**전체 MWO(180건) + SOP** 를 미리 임베딩해 둔 파일을 로드합니다. **API 호출이 전혀 없어** 빠르고 무료 한도·네트워크에 영향받지 않으며, 코퍼스가 전체라 검색 결과도 버전 1보다 풍부합니다.

> ⚠️ `corpus_vec.npy`·`q_vec.npy` 파일(강의 자료에 동봉)이 현재 폴더에 있어야 합니다. **버전 1 또는 버전 2 중 하나만** 실행하세요.

In [ ]:
import numpy as np

# 코퍼스 = MWO 기록 '전체'(180건) + SOP 6개 — 사전 임베딩(corpus_vec.npy)이 전체 기준이라 동일하게 맞춤
CORPUS = list(mwo_df["text"]) + MWO_SOP
q1 = "P-204 펌프에서 베어링 異音이 날 때 표준 조치 절차는?"
q2 = "지금까지 기록 중 가장 빈번한 정비 작업은 무엇이고 몇 건인가?"
QUESTIONS = [q1, q2]   # q_vec.npy 행 순서와 동일

# 미리 계산해 둔 임베딩 로드 — API 호출 없이 즉시 (오프라인·무료 한도 걱정 X)
corpus_vec = np.load("corpus_vec.npy")   # 전체 코퍼스 임베딩 (len == len(CORPUS))
q_vec = np.load("q_vec.npy")             # 질문 q1·q2 임베딩

def embed_query(question):
    """질문 임베딩 — 미리 저장한 q_vec 에서 조회(라이브 호출 없음)."""
    return q_vec[QUESTIONS.index(question)]

print(f"[버전 2] 사전 임베딩 로드 완료 — 코퍼스 {len(CORPUS)}개 (dim={corpus_vec.shape[1]})")

In [ ]:
# 검색 + 답변 — 버전 1·2 중 무엇을 실행했든 corpus_vec·q_vec·embed_query 를 그대로 사용
def rag_search(question, k=3):
    qv = embed_query(question)
    sims = corpus_vec @ qv / (np.linalg.norm(corpus_vec, axis=1) * np.linalg.norm(qv) + 1e-9)
    top = sims.argsort()[::-1][:k]
    return [(CORPUS[i], float(sims[i])) for i in top]

def rag_answer(question, k=3):
    hits = rag_search(question, k)
    context = NL.join(f"- {t}" for t, _ in hits)
    ans = ask(f"""아래 근거 문서만 사용해 질문에 답하세요. 근거에 없으면 '근거 없음'이라고 답하세요.

[근거 문서]
{context}

[질문] {question}""")
    return ans, hits

# 특정 사실형 질문 → RAG 강점
ans, hits = rag_answer(q1)
print("Q:", q1)
print()
print("검색된 근거:")
for t, s in hits: print(f"  ({s:.2f}) {t}")
print()
print("답변:")
print(ans)

## 5-2 · LLM 위키 — 지식 증류

로그 전체를 LLM 이 **장비별 고장 패턴 · 조치 SOP · FAQ** 구조화 위키(마크다운)로 선제 정제합니다.
벡터 청크가 약한 **전역·집계 질문**에 강하고, 사람이 검수 가능한 산출물이 남습니다.

In [ ]:
# 로그 전체 → 구조화 위키(markdown) 증류
all_logs = NL.join(f"- {t}" for t in mwo_df["text"])
wiki = ask(f"""당신은 제조 설비 신뢰성 엔지니어입니다.
아래 정비 기록 전체를 분석해 **정비 지식 위키(마크다운)**로 증류하세요. 포함:
## 1. 최빈 고장/조치 TOP3 (건수 포함)
## 2. 설비/부품별 패턴 (베어링·씰·모터 등)
## 3. 권장 조치 SOP 요약
## 4. FAQ 2~3개
기록에 근거해서만 작성. 추정은 '(추정)' 표기.

[정비 기록]
{all_logs}""")
from IPython.display import Markdown, display
display(Markdown(wiki))

In [ ]:
# 전역/집계 질문 — RAG(top-k 청크) vs 위키(전체 맥락) 비교
q2 = "지금까지 기록 중 가장 빈번한 정비 작업은 무엇이고 몇 건인가?"

# RAG: top-k 청크만 봐서 전체 집계에 약함
rag_ans, _ = rag_answer(q2, k=3)

# 위키: 이미 전역 집계가 정제돼 있어 강함
wiki_ans = ask(f"""아래 정비 지식 위키만 근거로 질문에 답하세요.

[위키]
{wiki}

[질문] {q2}""")

print("■ 전역 질문:", q2)
print()
print("[RAG · top-3 청크 기반]")
print(rag_ans)
print()
print("[LLM 위키 기반]")
print(wiki_ans)

### RAG vs LLM 위키 — 언제 무엇을?
| | 벡터 RAG | LLM 위키 / 증류 |
|--|----------|------------------|
| 강한 질문 | "이 증상 조치는?" (특정 사실) | "최빈 고장?" "추세는?" (전역·집계) |
| 셋업 | 벡터DB·청킹·인덱싱 | LLM 1회 정제 (벡터DB 불필요) |
| 산출물 | 검색 결과 | **사람이 검수 가능한 위키 문서** |
| 약점 | 전역·집계 약함 | 원문 추적성·최신성 관리 필요 |

> 실무는 **결합**: 위키로 전역 파악 → RAG 로 세부 근거 추적. (이를 **에이전트**가 스스로 오케스트레이션하게 발전)

---
# 🎁 보너스 — 문서 → MLLM 변환 (Beyond Logs)

현장 지식의 상당수는 **표·도면·스캔 PDF·한글 보고서**에 갇혀 있습니다.
멀티모달 LLM(Gemini)은 **이미지/PDF 를 구조화 마크다운·표**로 바꿔, 위의 5대 태스크 입력으로 흘려보냅니다.

In [ ]:
# 이미지(작업표준서 표 사진·점검표 스캔 등) 업로드 → 구조화 마크다운 표로 변환
from PIL import Image
import io

try:
    from google.colab import files
    up = files.upload()                      # 표/문서 이미지 1장 선택
    fname = next(iter(up))
    img = Image.open(io.BytesIO(up[fname]))
except Exception:
    print("Colab 업로드를 못 쓰는 환경입니다. img = Image.open('파일경로') 로 직접 여세요.")
    img = None

if img is not None:
    resp = client.models.generate_content(
        model=MODEL,
        contents=[
            "이 이미지의 표/문서 내용을 손실 없이 마크다운 표로 변환해줘. "
            "수치·단위·열 머리글을 그대로 보존하고, 판독이 어려운 셀은 [?]로 표기해줘.",
            img,
        ],
    )
    from IPython.display import Markdown, display
    display(Markdown(resp.text))

**📝 노코드 트랙 (문서 변환)** — **ChatGPT·Claude·Gemini 채팅**에서 **📎(이미지 첨부)** 로 표 사진을 올리고:

```text
이 이미지의 표/문서 내용을 손실 없이 마크다운 표로 변환해줘.
수치·단위·열 머리글을 그대로 보존하고, 판독이 어려운 셀은 [?]로 표기해줘.
```

> 변환된 마크다운을 그대로 실습1~3 프롬프트의 입력으로 넣으면, **종이/이미지 자료도 LLM 파이프라인**에 올라탑니다.

---
# ✅ 마무리

오늘 한 것 — **하나의 무료 Gemini 키**로:
1. **정규화** 표기 편차 18건 → 의미 단위 통합, 숨은 1위 작업 발견
2. **NER·분류·요약** 자유서술 → 구조화 JSON·자동 태깅(MWO·NCR)·알람 3줄 인과 요약 + MES 종합 리포트
3. **RAG vs LLM 위키** 특정 검색 vs 전역 집계, 상황별 선택 + 도메인 카드 8종
4. **보너스** 표 이미지/PDF → 구조화 마크다운으로 파이프라인 합류

### 다음 단계
- 본인 도메인 카드의 샘플을 **실제 로그 100건**으로 바꿔 5대 태스크를 돌려보기
- 결과를 **LLM 위키**로 증류 → 팀이 검수·버전관리하는 살아있는 문서로

> 질문은 언제든 — 이 노트북은 그대로 가져가 본인 데이터로 재사용하세요. 🙌